# 가장 기본적인 챗봇 만들기

LLM 을 노드 안에서 호출하는 **가장 단순한 LangGraph 챗봇**을 만든다.

흐름: `START → chatbot → END`. chatbot 노드가 LLM 을 호출해 답한다.
마지막에는 **대화 메모리(checkpointer)** 를 붙여 멀티턴 대화로 확장한다.

## 환경 변수 준비

이 노트북부터는 실제 LLM 을 호출하므로 API 키가 필요하다.
프로젝트 루트의 `.env` 파일에 키를 넣어두고 `load_dotenv()` 로 불러온다.

```
# .env  (.env.example 을 복사해서 작성)
OPENAI_API_KEY=sk-...
TAVILY_API_KEY=tvly-...   # 웹검색 노트북에서 필요
```

- OpenAI 키: <https://platform.openai.com/api-keys>
- Tavily 키: <https://app.tavily.com>

In [ ]:
import os
from dotenv import load_dotenv

# 프로젝트 루트의 .env 를 찾아 환경변수로 로드
load_dotenv()

# 필요한 키가 있는지 확인
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY 가 .env 에 없습니다"
print("환경변수 로드 완료:", "OPENAI_API_KEY")

## Step 1. 그래프 State 설정하기

메시지 리스트를 누적하는 State. `add_messages` 리듀서로 대화가 쌓이게 한다.

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

# [ch2 복습] State = 그래프 전체가 공유하며 노드 사이를 오가는 데이터 구조.
#            TypedDict 로 '이 State 는 어떤 키/타입을 갖는가' 를 선언한다.
class State(TypedDict):
    # [ch2 복습] Annotated[타입, 리듀서] = 이 필드를 '덮어쓰기' 대신 리듀서로 병합.
    #            add_messages = 메시지 리스트 전용 리듀서 (새 메시지를 누적, 같은 id 는 교체).
    #            → 노드가 새 메시지만 반환해도 대화가 차곡차곡 쌓인다.
    messages: Annotated[list, add_messages]

# [ch2 복습] StateGraph(스키마) = State 스키마로 그래프 빌더 생성.
graph_builder = StateGraph(State)

## Step 2. 챗봇 노드 추가하기

노드는 현재까지의 메시지를 LLM 에 넘기고, 그 응답을 messages 에 추가한다.

In [ ]:
from langchain_openai import ChatOpenAI

# ChatOpenAI = OpenAI 채팅 모델 래퍼 (이번 챕터의 새 개념)
llm = ChatOpenAI(model="gpt-4o")

# [ch2 복습] Node = (state) -> dict 함수. 현재 State 를 받아 '바꿀 필드만' dict 로 반환한다.
#            여기선 지금까지의 messages 를 LLM 에 넘기고, 그 응답을 messages 에 추가.
#            (add_messages 리듀서 덕분에 반환한 1개 메시지가 기존 대화에 누적됨)
def chatbot(state: State):
    return {"messages": [llm.invoke(state["messages"])]}

# [ch2 복습] add_node("이름", 함수) = 그래프에 노드 등록.
graph_builder.add_node("chatbot", chatbot)

## Step 3. 엣지 연결 후 컴파일

In [ ]:
# [ch2 복습] add_edge = 노드 간 흐름(엣지) 연결. START=진입점, END=종료점(둘 다 가상 노드).
graph_builder.add_edge(START, "chatbot")    # 진입점 → chatbot
graph_builder.add_edge("chatbot", END)       # chatbot → 종료
# [ch2 복습] compile() = 빌더를 '실행 가능한 그래프' 로 확정.
graph = graph_builder.compile()

## Step 4. 챗봇 실행하기

먼저 그래프 구조를 시각화한다.

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    print(graph.get_graph().draw_mermaid())

한 번 호출해보기. (반복 입력 루프 대신 단발 호출로 확인 — 루프 버전은 아래 주석 참고)

In [ ]:
# [ch2 복습] stream() = invoke 와 달리 '단계별 중간 결과' 를 실시간으로 내보낸다.
#            event 는 {노드이름: 그 노드의 반환값} 형태 (기본 stream_mode="updates").
def stream_graph_updates(user_input: str):
    # [ch2 복습] 입력 State 를 넣어 그래프 실행. dict 형태 메시지는 add_messages 가 변환해줌.
    for event in graph.stream({"messages": [{"role": "user", "content": user_input}]}):
        for value in event.values():
            print("Assistant:", value["messages"][-1].content)

stream_graph_updates("안녕하세요! 한 문장으로 자기소개 해줄래?")

터미널처럼 계속 대화하려면 아래처럼 입력 루프를 돌린다.
(`quit`/`exit`/`q` 입력 시 종료. 이 노트북에는 메모리가 없어 이전 대화를 기억하지 못한다.)

```python
while True:
    user_input = input("User: ")
    if user_input.lower() in ["quit", "exit", "q"]:
        print("Goodbye!")
        break
    stream_graph_updates(user_input)
```

## Step 5. 메모리 추가하기 (멀티턴)

`MemorySaver` 체크포인터를 붙이면 `thread_id` 단위로 대화 상태가 저장되어 **이전 대화를 기억**한다.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

# MemorySaver = 대화 상태를 메모리에 저장하는 체크포인터 (이번 챕터의 새 개념)
memory = MemorySaver()
# [ch2 복습] 같은 graph_builder 를 다시 compile — 이번엔 checkpointer 를 붙여서.
graph = graph_builder.compile(checkpointer=memory)

# thread_id = 대화 세션 식별자. 같은 id 면 이전 대화 상태가 이어진다.
config = {"configurable": {"thread_id": "1"}}

같은 `thread_id` 로 두 번 호출 — 두 번째 호출에서 앞 대화를 기억하는지 확인한다.

In [ ]:
# [ch2 복습] invoke() = 그래프를 한 번 실행하고 '최종 State' 를 반환 (stream 과 달리 한 번에).
res1 = graph.invoke(
    {"messages": [{"role": "user", "content": "내 이름은 스타리야. 기억해줘."}]},
    config=config,   # 같은 thread_id 로 호출해야 대화가 이어짐
)
print("1턴:", res1["messages"][-1].content)

res2 = graph.invoke(
    {"messages": [{"role": "user", "content": "내 이름이 뭐라고 했지?"}]},
    config=config,   # 1턴과 동일 thread_id → 앞 대화를 기억
)
print("2턴:", res2["messages"][-1].content)

## 정리

- 가장 단순한 챗봇 = `START → chatbot → END`, 노드 안에서 `llm.invoke`
- `ChatOpenAI(model="gpt-4o")` 로 LLM 준비
- **메모리** = `compile(checkpointer=MemorySaver())` + `config`의 `thread_id` → 멀티턴

다음: 외부 도구(웹검색)를 LLM 이 직접 호출하게 만드는 **Tool Calling**.